In [6]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
tf.random.set_seed(SEED)

In [7]:
# Load data 
BASE_DIR = "../data"
TRAIN_DIR = os.path.join(BASE_DIR, "Train")
VAL_DIR = os.path.join(BASE_DIR, "Validation")
TEST_DIR = os.path.join(BASE_DIR, "Test")

IMG_SIZE = (128, 128)
BATCH_SIZE = 32

In [8]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    color_mode="rgb",
    shuffle=True,
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    color_mode="rgb",
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    color_mode="rgb",
    shuffle=False
)

class_names = train_ds.class_names
print(class_names)

Found 10000 files belonging to 2 classes.
Found 800 files belonging to 2 classes.
Found 992 files belonging to 2 classes.
['WithMask', 'WithoutMask']


In [9]:
# Build model with transfer learning (augmentation + rescale inside the model)
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(128, 128, 3))
base_model.trainable = False  # Freeze base model

model = models.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomTranslation(0.1, 0.1),
    layers.Rescaling(1./127.5, offset=-1),  # MobileNetV2 expects [-1, 1]
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])


In [5]:
# Fine-tune một số layer của base model 
base_model.trainable = True
for layer in base_model.layers[:-30]:  # Freeze tất cả trừ 30 layer cuối
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='binary_crossentropy',
              metrics=['accuracy'])

history_finetune = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

Epoch 1/5


c:\Users\HOME\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


313/313 ━━━━━━━━━━━━━━━━━━━━ 104s 301ms/step - accuracy: 0.8779 - loss: 0.2922 - val_accuracy: 0.9137 - val_loss: 0.1867
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 98s 313ms/step - accuracy: 0.9716 - loss: 0.0908 - val_accuracy: 0.9725 - val_loss: 0.0675
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 98s 312ms/step - accuracy: 0.9780 - loss: 0.0632 - val_accuracy: 0.9900 - val_loss: 0.0247
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 106s 339ms/step - accuracy: 0.9813 - loss: 0.0513 - val_accuracy: 0.9975 - val_loss: 0.0101
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 105s 334ms/step - accuracy: 0.9850 - loss: 0.0426 - val_accuracy: 0.9975 - val_loss: 0.0069


In [11]:
print(model.compiled, history_finetune.history["val_accuracy"][-1])

False 0.9975000023841858


In [10]:
# Evaluate
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test accuracy: {test_acc:.4f}")

y_true = np.concatenate([y.numpy() for _, y in test_ds]).flatten().astype(int)
y_pred = (model.predict(test_ds) >= 0.5).astype(int).flatten()
print(classification_report(y_true, y_pred, target_names=class_names))
print(confusion_matrix(y_true, y_pred))

ValueError: You must call `compile()` before using the model.